## Bartłomiej Stępniewski lab 7

### 0. Libraries import, model training

In [2]:
import time
import torch
from transformers import AutoTokenizer, AutoModel

MODEL_NAME = "sentence-transformers/multi-qa-mpnet-base-cos-v1"

In [3]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME)
text = "Nas będą z tego głównie interesowały „przepustowości łączy”, czyli ile bitów danych jesteśmy w stanie przesłać przez sieć komputerową w czasie jednej sekundy"
inputs = tokenizer(text, padding=True, truncation=True, return_tensors="pt")

In [4]:
### Check to be certain that few first excersises run on CPU
print(next(model.parameters()).device)

cpu


#### Ex. 1

#### 1.1. Pure PyTorch time

In [5]:
times_pure_torch = []

for _ in range(100):
    start_time = time.time()
    model(**inputs)
    end_time = time.time()
    times_pure_torch.append(end_time - start_time)

print("Avg: ", sum(times_pure_torch) / len(times_pure_torch))

Avg:  0.0324771237373352


#### 1.2. model.eval() time

In [6]:
model.eval()

MPNetModel(
  (embeddings): MPNetEmbeddings(
    (word_embeddings): Embedding(30527, 768, padding_idx=1)
    (position_embeddings): Embedding(514, 768, padding_idx=1)
    (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): MPNetEncoder(
    (layer): ModuleList(
      (0-11): 12 x MPNetLayer(
        (attention): MPNetAttention(
          (attn): MPNetSelfAttention(
            (q): Linear(in_features=768, out_features=768, bias=True)
            (k): Linear(in_features=768, out_features=768, bias=True)
            (v): Linear(in_features=768, out_features=768, bias=True)
            (o): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (intermediate): MPNetIntermediate(
          (dense): Linear(in_

In [7]:
times_eval = []
for _ in range(100):
    start_time = time.time()
    with torch.no_grad():
        model(**inputs)
    end_time = time.time()
    times_eval.append(end_time - start_time)

print("Avg eval: ", sum(times_eval) / len(times_eval))

Avg eval:  0.02960919141769409


#### 1.3. no_grad() time

In [8]:
times_no_grad_eval = []
for _ in range(100):
    start_time = time.time()
    with torch.no_grad():
        model(**inputs)
    end_time = time.time()
    times_no_grad_eval.append(end_time - start_time)

print("Avg no_grad eval: ", sum(times_no_grad_eval) / len(times_no_grad_eval))

Avg no_grad eval:  0.029692463874816895


#### 1.4. Inference_mode() time

In [9]:
times_inference_mode = []
for _ in range(100):
    start_time = time.time()
    with torch.inference_mode():
        model(**inputs)
    end_time = time.time()
    times_inference_mode.append(end_time - start_time)

print("Avg inference_mode: ", sum(times_inference_mode) / len(times_inference_mode))

Avg inference_mode:  0.029661107063293456


#### 1.5. Comparison

In [10]:
avg_pure = sum(times_pure_torch) / len(times_pure_torch)
avg_eval = sum(times_eval) / len(times_eval)
avg_no_grad = sum(times_no_grad_eval) / len(times_no_grad_eval)
avg_inference = sum(times_inference_mode) / len(times_inference_mode)

eval_multiplier = avg_pure / avg_eval
no_grad_multiplier = avg_pure / avg_no_grad
inference_multiplier = avg_pure / avg_inference

print(f"eval is faster {eval_multiplier:.3f}x than pure")
print(f"no_grad is faster {no_grad_multiplier:.3f}x than pure")
print(f"inference_mode is faster {inference_multiplier:.3f}x than pure")

eval is faster 1.097x than pure
no_grad is faster 1.094x than pure
inference_mode is faster 1.095x than pure


### 2. Ex 2. - torch.compile()

In [11]:
# Sth dont works, probably because I have Windows on my stationary computer.
# I will this mistake for the next lab :)

import torch._dynamo
torch._dynamo.config.suppress_errors = True

In [12]:
## Torch compile
device = torch.device("cpu")
model.to(device)
compiled_model = torch.compile(model)

#warm-up
warm_start = time.time()
with torch.inference_mode():
    compiled_model(**inputs)
warm_end = time.time()
print("Warm-up time: ", warm_end - warm_start)

### compiled model time

times_compiled = []

for _ in range(100):
    start_time = time.time()
    with torch.inference_mode():
        compiled_model(**inputs)
    end_time = time.time()
    times_compiled.append(end_time - start_time)

print("Avg compiled inference_mode: ", sum(times_compiled) / len(times_compiled))

W1125 17:10:12.258000 24112 Lib\site-packages\torch\_dynamo\convert_frame.py:1708] WON'T CONVERT forward c:\Users\bstepniewski_vr\programowanie\pytong\MLOps_lab07\.venv\Lib\site-packages\transformers\models\mpnet\modeling_mpnet.py line 449 
W1125 17:10:12.258000 24112 Lib\site-packages\torch\_dynamo\convert_frame.py:1708] due to: 
W1125 17:10:12.258000 24112 Lib\site-packages\torch\_dynamo\convert_frame.py:1708] Traceback (most recent call last):
W1125 17:10:12.258000 24112 Lib\site-packages\torch\_dynamo\convert_frame.py:1708]   File "c:\Users\bstepniewski_vr\programowanie\pytong\MLOps_lab07\.venv\Lib\site-packages\torch\_dynamo\convert_frame.py", line 1625, in __call__
W1125 17:10:12.258000 24112 Lib\site-packages\torch\_dynamo\convert_frame.py:1708]     result = self._inner_convert(
W1125 17:10:12.258000 24112 Lib\site-packages\torch\_dynamo\convert_frame.py:1708]              ^^^^^^^^^^^^^^^^^^^^
W1125 17:10:12.258000 24112 Lib\site-packages\torch\_dynamo\convert_frame.py:1708]   F

Warm-up time:  11.850373029708862
Avg compiled inference_mode:  0.02914051532745361


In [13]:
print("Compiled is ", avg_inference / (sum(times_compiled) / len(times_compiled)), "time faster than inference_mode")
print("Compiled is ", avg_pure / (sum(times_compiled) / len(times_compiled)), "time faster than pure")
print("Compiled is ", avg_eval / (sum(times_compiled) / len(times_compiled)), "time faster than eval")
print("Compiled is ", avg_no_grad / (sum(times_compiled) / len(times_compiled)), "time faster than no_grad")

Compiled is  1.0178648774735082 time faster than inference_mode
Compiled is  1.1145006659075152 time faster than pure
Compiled is  1.0160833151018072 time faster than eval
Compiled is  1.0189409329643284 time faster than no_grad


### Ex. 3 - quantization

In [14]:
### one more check

print(next(model.parameters()).device)

cpu


#### 3.1. Times comparison

In [15]:
from torch.ao.quantization import quantize_dynamic

quantized_model = quantize_dynamic(
    model,
    {torch.nn.Linear},
    dtype=torch.qint8
)

### quantized model time

times_quantized = []

for _ in range(100):
    start_time = time.time()
    with torch.inference_mode():
        quantized_model(**inputs)
    end_time = time.time()
    times_quantized.append(end_time - start_time)

avg_quantized = sum(times_quantized) / len(times_quantized)
print("Avg quantized inference_mode: ", avg_quantized)

print("Quantized is ", avg_inference / avg_quantized, "times faster than inference_mode")
print("Quantized is ", avg_pure / avg_quantized, "times faster than pure")
print("Quantized is ", avg_eval / avg_quantized, "times faster than eval")
print("Quantized is ", avg_no_grad / avg_quantized, "times faster than no_grad")
print("Quantized is ", (sum(times_compiled) / len(times_compiled)) /
        avg_quantized, "times faster than compiled")

C:\Users\bstepniewski_vr\AppData\Local\Temp\ipykernel_24112\3607582116.py:3: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  quantized_model = quantize_dynamic(


Avg quantized inference_mode:  0.010399465560913085
Quantized is  2.852176094007775 times faster than inference_mode
Quantized is  3.1229608432381477 times faster than pure
Quantized is  2.847183948469595 times faster than eval
Quantized is  2.855191326986794 times faster than no_grad
Quantized is  2.8021166238561053 times faster than compiled


#### 3.2. Size comparison

In [16]:
import os
def get_size(model, label):
    torch.save(model.state_dict(), f"{label}_model.pth")
    size = os.path.getsize(f"{label}_model.pth") / (1024 * 1024)
    os.remove(f"{label}_model.pth")
    return size

size_original = get_size(model, "original")
size_quantized = get_size(quantized_model, "quantized")

print(f"Original model size: {size_original:.2f} MB")
print(f"Quantized model size: {size_quantized:.2f} MB")
print(f"Quantized model is {size_original / size_quantized:.2f} times smaller than original")

Original model size: 417.73 MB
Quantized model size: 173.10 MB
Quantized model is 2.41 times smaller than original


#### 3.3. Remarks

Yes, in this case quantization is really useful and this is the first method which significantly speed up the model

### 4. Ex 4

#### 4.0 - transfering model to GPU

In [38]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
gpu_model = model.to(device)
inputs_gpu = {k: v.to(device) for k, v in inputs.items()}

#### 4.1. [GPU] torch.compile()

In [39]:
gpu_model_compiled = torch.compile(gpu_model)

times_compiled_gpu = []

for _ in range(100):
    start_time = time.time()
    with torch.inference_mode():
        gpu_model_compiled(**inputs_gpu)
    end_time = time.time()
    times_compiled_gpu.append(end_time - start_time)

avg_compiled_gpu = sum(times_compiled_gpu) / len(times_compiled_gpu)
print("Avg compiled GPU inference_mode: ", avg_compiled_gpu)

Avg compiled GPU inference_mode:  0.0064980053901672365


#### 4.2. max-autotune

In [40]:
model_max_autotune = torch.compile(gpu_model, mode="max-autotune")

times_max_autotune = []

for _ in range(100):
    start_time = time.time()
    with torch.inference_mode():
        model_max_autotune(**inputs_gpu)
    end_time = time.time()
    times_max_autotune.append(end_time - start_time)

avg_max_autotune = sum(times_max_autotune) / len(times_max_autotune)
print("Avg max-autotune GPU inference_mode: ", avg_max_autotune)

Avg max-autotune GPU inference_mode:  0.004980552196502686


#### 4.3. max-autotune no cuda graphs

In [41]:
model_no_cuda_graphs = torch.compile(model, mode="max-autotune-no-cudagraphs")

times_no_cuda_graphs = []

for _ in range(100):
    start_time = time.time()
    with torch.inference_mode():
        model_no_cuda_graphs(**inputs_gpu)
    end_time = time.time()
    times_no_cuda_graphs.append(end_time - start_time)

avg_no_cuda_graphs = sum(times_no_cuda_graphs) / len(times_no_cuda_graphs)

print("Avg max-autotune no cuda graphs GPU inference_mode: ", avg_no_cuda_graphs)

Avg max-autotune no cuda graphs GPU inference_mode:  0.005230762958526611


#### Longer text

In [42]:
text_longer = "Komitet Bezpieczeństwa Państwowego przy Radzie Ministrów ZSRR,utworzony w 1954 po likwidacji MGB organ państwowy przy Radzie Ministrów ZSRR " \
"kierujący siłami bezpieczeństwa wewnętrznego (policja polityczna, wojska wewnętrzne i wojska ochrony pogranicza) i zewnętrznego (wywiad i kontrwywiad, " \
"aparat dywersyjno-szpiegowski). Jedna z dwóch służb specjalnych ZSRR. Stosował inwigilację ludności, dezinformację, zwalczał rzeczywistych i domniemanych " \
"przeciwników KPZR i niezależny obieg informacji. KGB istniał od marca 1954 do października 1991. Był następcą m.in. CzeKa i NKWD. " \
"W propagandzie komunistycznej często określany jako „tarcza i miecz partii”. W trakcie pieriestrojki i w czasie rozpadu ZSRR był stopniowo dzielony, " \
"a wiele komórek organizacyjnych zlikwidowano. Główne komórki organizacyjne KGB zostały przekształcone w Federacji Rosyjskiej w szereg instytucji: " \
"Służba Wywiadu Zagranicznego Federacji Rosyjskiej, Federalna Służba Bezpieczeństwa Federacji Rosyjskiej, Federalna Służba Ochrony Federacji Rosyjskiej. " \
"Do głównych zadań KGB należało gromadzenie informacji wywiadowczych poza granicami Związku Radzieckiego, ochrona kontrwywiadowcza armii, marynarki wojennej i " \
"lotnictwa oraz samego Komitetu, ochrona granic państwowych ZSRR, nadzór nad obiektami nuklearnymi i większymi ośrodkami przemysłowymi tego kraju, " \
"ochrona budynków rządowych oraz dostojników partyjnych w kraju i poza jego granicami, w tym ambasad i konsulatów. W skład KGB wchodziło dziewięć zarządów głównych, " \
"z których największą rolę odgrywały: I (wywiad), II (kontrwywiad), III (kontrwywiad wojskowy), V (walka ideologiczna i zwalczanie dysydentów), VI " \
"(wywiad i kontrwywiad gospodarczy), VII (inwigilacja) i IX (ochrona dostojników państwowych). Oprócz tego poza strukturami zarządów głównych istniały specjalne " \
"oddziały zabójców działających na całym świecie (tzw. mokryje dieła)."

inputs_longer = tokenizer(text_longer, padding=True, truncation=True, return_tensors="pt")
inputs_longer_gpu = {k: v.to(device) for k, v in inputs_longer.items()}

In [43]:
times_max_autotune_longer = []

for _ in range(100):
    start_time = time.time()
    with torch.inference_mode():
        model_max_autotune(**inputs_longer_gpu)
    end_time = time.time()
    times_max_autotune_longer.append(end_time - start_time)

avg_max_autotune_longer = sum(times_max_autotune_longer) / len(times_max_autotune_longer)
print("Avg max-autotune longer input GPU inference_mode: ", avg_max_autotune_longer)

Avg max-autotune longer input GPU inference_mode:  0.010147106647491456


In [44]:
times_no_cuda_graphs_longer = []

for _ in range(100):
    start_time = time.time()
    with torch.inference_mode():
        model_no_cuda_graphs(**inputs_longer_gpu)
    end_time = time.time()
    times_no_cuda_graphs_longer.append(end_time - start_time)

avg_no_cuda_graphs_longer = sum(times_no_cuda_graphs_longer) / len(times_no_cuda_graphs_longer)
print("Avg max-autotune no cuda graphs longer input GPU inference_mode: ", avg_no_cuda_graphs_longer)

Avg max-autotune no cuda graphs longer input GPU inference_mode:  0.010065951347351075


#### 4.5. Comparison

In [48]:
cpu_inference = avg_inference # selected from inference_mode on CPU
print("--------TIMES-------------")
print(f"CPU inference_mode time: {cpu_inference:.6f} s")
print(f"Pure compiled GPU inference time: {avg_compiled_gpu:.6f} s")
print(f"Max-autotune GPU inference time: {avg_max_autotune:.6f} s")
print(f"Max-autotune no cuda graphs GPU inference time: {avg_no_cuda_graphs:.6f} s")
print("\n------ TO CPU ------------")
print(f"Pure compiled GPU is {(cpu_inference / avg_compiled_gpu):.2f} times faster than CPU inference_mode")
print(f"Max-autotune GPU is {(cpu_inference / avg_max_autotune):.2f} times faster than CPU inference_mode")
print(f"Max-autotune no cuda graphs GPU is {(cpu_inference / avg_no_cuda_graphs):.2f} times faster than CPU inference_mode")
print("\n---- LONGER INPUTS -------")
print(f"Pure compiled longer input GPU inference time: {avg_compiled_gpu:.6f} s")
print(f"Max-autotune longer input GPU inference time: {avg_max_autotune_longer:.6f} s")
print(f"Max-autotune no cuda graphs longer input GPU inference time: {avg_no_cuda_graphs_longer:.6f} s")
                                                

--------TIMES-------------
CPU inference_mode time: 0.029661 s
Pure compiled GPU inference time: 0.006498 s
Max-autotune GPU inference time: 0.004981 s
Max-autotune no cuda graphs GPU inference time: 0.005231 s

------ TO CPU ------------
Pure compiled GPU is 4.56 times faster than CPU inference_mode
Max-autotune GPU is 5.96 times faster than CPU inference_mode
Max-autotune no cuda graphs GPU is 5.67 times faster than CPU inference_mode

---- LONGER INPUTS -------
Pure compiled longer input GPU inference time: 0.006498 s
Max-autotune longer input GPU inference time: 0.010147 s
Max-autotune no cuda graphs longer input GPU inference time: 0.010066 s


#### 4.5. Remarks

As we can see:
* Max-autotune is slightly faster than default compilation
* no cuda graphs are delicately slower than standard max-autotune for shorter text, but for longer input no cuda graphs compilation is minimally faster - this may indicate that no cuda graphs maybe useful for longer outputs

### 5. Ex 5
<i>Remark: I assume that we are comparing vanilla(uncompiled) models</i>

#### 5.1. Checking GPU capability

In [49]:
capability = torch.cuda.get_device_capability()
print(f"CUDA device capability: {capability}")

# Tensor Cores are available on NVidia GPUs with CUDA >= 7 (e.g. Volta, Turing, Ampere, Hopper)
if capability >= (7, 0):
    print("Tensor Cores available: fast float16 supported.")
else:
    print("Tensor Cores not available: float16 may be slow or unsupported.")

CUDA device capability: (12, 0)
Tensor Cores available: fast float16 supported.


#### 5.2. gpu model uncompiled(for comparison)

In [76]:
gpu_model_uncompiled_times = []

for _ in range(100):
    start_time = time.time()
    with torch.inference_mode():
        gpu_model(**inputs_gpu)
    end_time = time.time()
    gpu_model_uncompiled_times.append(end_time - start_time)

avg_uncompiled_gpu = sum(gpu_model_uncompiled_times) / len(gpu_model_uncompiled_times)
print("Avg uncompiled GPU inference_mode: ", avg_uncompiled_gpu)

Avg uncompiled GPU inference_mode:  0.004815080165863037


#### 5.3. GPU Model manual half

In [77]:
gpu_model_half = gpu_model.half().to('cuda')

times_half_gpu = []
for _ in range(100):
    start_time = time.time()
    with torch.inference_mode():
        outputs = gpu_model_half(inputs_gpu['input_ids'], inputs_gpu['attention_mask'].half())
    end_time = time.time()
    times_half_gpu.append(end_time - start_time)

avg_half_gpu = sum(times_half_gpu) / len(times_half_gpu)
print("Avg half precision GPU inference_mode: ", avg_half_gpu)

Avg half precision GPU inference_mode:  0.004774060249328613


5.4. GPU Model autocast(with eval())

In [78]:
gpu_model_autocast = gpu_model.eval()

times_autocast_gpu = []

for _ in range(100):
    start_time = time.time()
    with torch.inference_mode():
        with torch.cuda.amp.autocast():
            gpu_model_autocast(**inputs_gpu)
    end_time = time.time()
    times_autocast_gpu.append(end_time - start_time)
avg_autocast_gpu = sum(times_autocast_gpu) / len(times_autocast_gpu)
print("Avg autocast GPU inference_mode: ", avg_autocast_gpu)

C:\Users\bstepniewski_vr\AppData\Local\Temp\ipykernel_24112\2948971611.py:8: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


Avg autocast GPU inference_mode:  0.006253855228424072


#### 5.5. Comparison

In [80]:
print(f"Uncompiled GPU inference time: {avg_uncompiled_gpu:.6f} s")
print(f"Half precision GPU inference time: {avg_half_gpu:.6f} s")
print(f"Autocast GPU inference time: {avg_autocast_gpu:.6f} s")

Uncompiled GPU inference time: 0.004815 s
Half precision GPU inference time: 0.004774 s
Autocast GPU inference time: 0.006254 s


Maybe I am doing something wrong(no improvement with half-precision), but for this case I would use standard FP32 model. Maybe I would run some quick test with FP16

### 6. Ex. 6 ONNX

#### 6.0. Preparations

In [116]:
import torch.onnx

# Put the model in eval mode and move to CPU
model_cpu = model.eval().cpu()

# Example input for tracking (for onnx export)
sample_input = tokenizer(
    "This is a sample input text for ONNX export.",
    padding=True,
    truncation=True,
    return_tensors="pt",
)

# Export to ONNX format
torch.onnx.export(
    model_cpu,
    (sample_input["input_ids"], sample_input["attention_mask"]),
    "model.onnx",
    opset_version=17,
    input_names=["input_ids", "attention_mask"],
    output_names=["output"],
    dynamic_axes={
        "input_ids": {0: "batch_size", 1: "sequence_length"},
        "attention_mask": {0: "batch_size", 1: "sequence_length"},
        "output": {0: "batch_size"},
    },
    dynamo=False
)

inputs_onnx = {
    "input_ids": sample_input["input_ids"],
    "attention_mask": sample_input["attention_mask"],
}

C:\Users\bstepniewski_vr\AppData\Local\Temp\ipykernel_24112\1690946289.py:15: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter will be the default. To switch now, set dynamo=True in torch.onnx.export. This new exporter supports features like exporting LLMs with DynamicCache. We encourage you to try it and share feedback to help improve the experience. Learn more about the new export logic: https://pytorch.org/docs/stable/onnx_dynamo.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html.
  torch.onnx.export(


#### 6.1. Online

In [130]:
import onnxruntime as ort

options = ort.SessionOptions()
options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL

ort_session = ort.InferenceSession(
    "model.onnx", sess_options=options, providers=["CPUExecutionProvider"]
)

In [135]:
### Cold start online

time_first_online = time.time()
outputs = ort_session.run(None, inputs_onnx)
time_second_online = time.time()
cold_start_time_online = time_second_online - time_first_online
print("Cold start time online: ", cold_start_time_online)

Cold start time online:  0.011400699615478516


In [136]:
onnx_online_times = []

for _ in range(100):
    start_time = time.time()
    outputs = ort_session.run(None, inputs_onnx)
    end_time = time.time()
    onnx_online_times.append(end_time - start_time)
avg_onnx_online = sum(onnx_online_times) / len(onnx_online_times)
print("Avg ONNX online inference time: ", avg_onnx_online)

Avg ONNX online inference time:  0.00907245397567749


#### 6.2. Offline

In [137]:
sess_options = ort.SessionOptions()

# Choose the optimization level for the offline pass
sess_options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_EXTENDED

# Save the optimized model to this path
sess_options.optimized_model_filepath = "model_optimized.onnx"

# Create InferenceSession, which will perform offline optimization and save the optimized model
ort.InferenceSession("model.onnx", sess_options)

sess_options = ort.SessionOptions()
sess_options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_DISABLE_ALL

ort_session_optimized = ort.InferenceSession(
    "model_optimized.onnx", 
    sess_options=sess_options, 
    providers=['CPUExecutionProvider']
)

In [139]:
### cold start offline

time_first_offline = time.time()
outputs = ort_session_optimized.run(None, inputs_onnx)
time_second_offline = time.time()
cold_start_time_offline = time_second_offline - time_first_offline
print("Cold start time offline: ", cold_start_time_offline)

Cold start time offline:  0.01659989356994629


In [140]:
onnx_offline_times = []

for _ in range(100):
    start_time = time.time()
    outputs = ort_session_optimized.run(None, inputs_onnx)
    end_time = time.time()
    onnx_offline_times.append(end_time - start_time)
avg_onnx_offline = sum(onnx_offline_times) / len(onnx_offline_times)
print("Avg ONNX offline inference time: ", avg_onnx_offline)

Avg ONNX offline inference time:  0.010441038608551025


#### 6.3. Comparison

In [142]:
print(f"Cold start ONNX online: {cold_start_time_online:.6f}")
print(f"Cold start ONNX offline: {cold_start_time_offline:.6f}")
print(f"Avg ONNX online inference time: {avg_onnx_online:.6f}")
print(f"Avg ONNX offline inference time: {avg_onnx_offline:.6f}")

Cold start ONNX online: 0.011401
Cold start ONNX offline: 0.016600
Avg ONNX online inference time: 0.009072
Avg ONNX offline inference time: 0.010441


<i>The Docker task is not done</i>